In [6]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")

#Summarization Middleware- Automatically summarizes message history when the user reach token limit

In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent = create_agent(
    model  = "groq:openai/gpt-oss-20b",
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware(
            model  = "groq:openai/gpt-oss-20b",
            trigger= ('messages', 10),
            keep = ('messages', 4)
        )
    ]
)

In [8]:
config = {"configurable": {"thread_id":"test-1"}}

In [9]:
questions=[
    "what is 2 + 2",
    "what is 10 /290",
    "what is 10 * 100",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")

Messages: {'messages': [HumanMessage(content='what is 2 + 2', additional_kwargs={}, response_metadata={}, id='ca6f14e3-b6ba-4474-870f-7e2d72a3a6aa'), AIMessage(content='2\u202f+\u202f2\u202f=\u202f4.', additional_kwargs={'reasoning_content': 'User asks simple math: 2+2 = 4. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 78, 'total_tokens': 114, 'completion_time': 0.040674311, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.004251403, 'prompt_tokens_details': None, 'queue_time': 0.314451746, 'total_time': 0.044925714}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_565badff47', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a115bc-f9be-7e10-84af-a2fc9d00b72c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 36, 'total_tokens': 114, 'output_token_details': {'reasoning': 17}})]}
Messages: {'me

#Summarization through token size


In [ ]:
from langchain_core.tools import tool

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long response to use more tokens"""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, buisness center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model = "groq:openai/gpt-oss-20b",
    tools = [search_hotel],
    checkpointer = InMemorySaver(),
    
)